In [1]:
import os
import sys
from pathlib import Path

ROOT = Path(os.path.abspath('')).resolve().parents[2]
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))


import nvitk as nv
from nvitk import db
import pandas as pd

In [2]:
from nvitk.db.repo import DataRepo
from nvitk.db.xnat import XnatConnectionConfig, sync_xnat_project

PROJECT="PESA_Brain"

In [8]:


repo = DataRepo("~/nvitk/dataset", auto_scaffold=True)
config = XnatConnectionConfig(
    server='https://xnat.cnic.es',
    project={PROJECT},
    netrc_file='~/.netrc',
)
config


XnatConnectionConfig(server='https://xnat.cnic.es', project={'PESA_Brain'}, user=None, password=None, netrc_file='~/.netrc', verify=True, default_timeout=300)

In [7]:

frames = sync_xnat_project(
    repo,
    config,
    subjects=["PESA10407076", "PESA16016004"],
    requested_sequences="TOF,4DFLOW_AP,4DFLOW_FH,4DFLOW_RL",  # optional
    download_root="/data_local/LabVF/PESA-Brain/DATA/BatchTest/DICOM_XNAT_TEST/",          # optional
    download_dicoms=True,                # True to populate assets + local DICOM paths
    build_sqlite_index=True,
)
frames

SSLError: HTTPSConnectionPool(host='xnat.cnic.es', port=443): Max retries exceeded with url: / (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate (_ssl.c:1016)')))

In [ ]:
from nvitk.db import DataRepo, sync_xnat_project

repo = DataRepo("~/nvitk/dataset", auto_scaffold=True)

frames = sync_xnat_project(
    repo,
    config,
    subjects=["PESA10407076", "PESA16016004"],          # or None for all subjects in project
    requested_sequences="TOF,4DFLOW_AP,4DFLOW_FH,4DFLOW_RL",  # optional filter
    download_root="/data_local/LabVF/PESA-Brain/DATA/BatchTest/DICOM_XNAT_TEST/",      # required for downloads
    download_dicoms=False,                  # DICOM assets under .../{subject}/{sequence}/
    download_niftis=False,                  # NIfTI under .../{subject}/{sequence}/nifti/
    nifti_resource_label="DICOM",
    nifti_download_root=None,              # None → same base as download_root logic in code
    skip_existing=False,
    build_sqlite_index=True,
)

# frames["subjects"], frames["sessions"], frames["scans"], frames["assets"] — may be empty if nothing matched
frames

In [ ]:
# Preferred helper (defaults use_sqlite=True if index exists)
assets_df = repo.assets()

# Same data, explicit filters (column names match the catalog)
assets_df = repo.assets(filters={"modality": "4dflow", "asset_type": "DICOM"})
assets_df = repo.assets(filters={"source": "xnat"})
assets_df = repo.assets(filters={"subject_uid": "PESA10407076"})

# Lower-level equivalent
assets_df = repo.get("assets", filters={"modality": "tof"})

In [ ]:
from nvitk.db import connect_xnat

with connect_xnat(config) as session:
    project = session.projects[config.project]
    subj = project.subjects["PESA10407076"]
    # ... use pyxnat/xnat API directly